[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-support_triage/examples/support_triage/support_triage.ipynb)

# Route support tickets by confidence

A ticket has to reach a team, and someone has to decide when the model should hand it to a person. An LLM would write a label you then parse. FRIDA-Decisions scores the team (`choice`), the urgency (`score`) and two yes/no checks in one encoder pass, and returns probabilities. The threshold that sends a ticket to an operator lives in code: changing it does not rewrite the questions.

Option ids are short and neutral (`auth`, `billing`). The model sees `"<id>: <description>"`, so the meaning sits in the description.

Patterns: [Intent routing](https://docs.typesafe.ai/patterns/intent-routing) and [Confidence-gated routing](https://docs.typesafe.ai/patterns/confidence-routing). `confidence` is reported for `choice` and `score`, not for `noul`.


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
TEAMS = {
    "auth": "доступ к аккаунту, вход, пароль, личные данные",
    "billing": "оплата, списания, возвраты денег, подписки",
    "shipping": "доставка, курьер, сроки и адрес получения заказа",
    "product": "качество, комплектность и характеристики товара",
}

URGENCY = [
    "Очень низкая: работа не страдает, ответить можно когда угодно.",
    "Низкая: неудобство небольшое, ответить можно на этой неделе.",
    "Средняя: работа затруднена, но обходной путь есть.",
    "Высокая: работа заблокирована, обходного пути нет.",
    "Критическая: потери растут с каждым часом.",
]

# Starting point, not a measured constant. The sweep below is the measurement.
DEFAULT_CONFIDENCE = 0.7
HIGH_URGENCY = 3.0
UPSET = 0.6


def request_for(text: str) -> dict:
    return {
        "state": text,
        "questions": {
            "team": {"type": "choice", "instructions": "В какую команду направить обращение?",
                     "criteria": TEAMS},
            "urgency": {"type": "score", "instructions": "Оцени срочность обращения по шкале ниже.",
                        "criteria": URGENCY},
            "angry": {"type": "noul", "instructions": "Автор раздражён?"},
            "churn": {"type": "noul", "instructions": "Автор грозит уйти или больше не покупать?"},
        },
    }


def route(answers: dict, min_confidence: float) -> dict:
    """Team queue when the choice is confident enough, otherwise a person.

    Urgency and the two yes/no checks only set the priority. They do not pick the team.
    """
    team = answers["team"]
    urgent = answers["urgency"]["score"] >= HIGH_URGENCY
    upset = answers["angry"]["noul"] >= UPSET or answers["churn"]["noul"] >= UPSET
    auto = team["confidence"] >= min_confidence
    return {
        "queue": team["choice"] if auto else "operator",
        "auto": auto,
        "priority": "high" if urgent or upset else "normal",
        "confidence": team["confidence"],
    }


def evaluate(scored: list[dict], min_confidence: float) -> dict:
    decided = []
    correct = 0
    for row in scored:
        decision = route(row["answers"], min_confidence)
        if not decision["auto"]:
            continue
        decided.append(decision)
        correct += decision["queue"] == row["team"]
    return {
        "n": len(scored),
        "auto": len(decided),
        "coverage": len(decided) / len(scored),
        "accuracy": (correct / len(decided)) if decided else None,
    }


## The threshold lives in code

Every ticket is scored once. Coverage is the share sent to a team without a person.
Accuracy is only over those tickets: a ticket handed to an operator is not counted as a miss.


In [4]:
def run(judge, rows: list[dict]) -> None:
    responses = judge.judge_batch([request_for(row["text"]) for row in rows])
    scored = []
    print(f"{'result':8} {'queue':10} {'pri':6} conf  urgency angry churn  gold")
    for row, response in zip(rows, responses):
        decision = route(response["answers"], DEFAULT_CONFIDENCE)
        answers = response["answers"]
        scored.append({"team": row["team"], "answers": answers})
        if decision["auto"] and decision["queue"] == row["team"]:
            mark = "ok"
        elif decision["auto"]:
            mark = "miss"
        else:
            mark = "hold"
        print(f"{mark:8} {decision['queue']:10} {decision['priority']:6} "
              f"{decision['confidence']:.2f}  {answers['urgency']['score']:.2f}    "
              f"{answers['angry']['noul']:.2f}  {answers['churn']['noul']:.2f}   {row['team']}")
        if response["usage"]["state_truncated"]:
            print("  state truncated: trained range is 512 tokens, the default cut is 384")

    print(f"\nthreshold {DEFAULT_CONFIDENCE}:")
    summary = evaluate(scored, DEFAULT_CONFIDENCE)
    accuracy = "n/a" if summary["accuracy"] is None else f"{summary['accuracy']:.0%}"
    print(f"  auto {summary['auto']}/{summary['n']} ({summary['coverage']:.0%}), "
          f"accuracy on auto {accuracy}")
    print("\nthreshold sweep (accuracy is only over tickets the model routes itself):")
    for threshold in (0.0, 0.5, 0.7, 0.8, 0.9, 0.95):
        summary = evaluate(scored, threshold)
        accuracy = "n/a" if summary["accuracy"] is None else f"{summary['accuracy']:.0%}"
        print(f"  {threshold:.2f}  coverage {summary['coverage']:.0%}  accuracy {accuracy}")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-support-triage/examples/support_triage/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


result   queue      pri    conf  urgency angry churn  gold
ok       auth       normal 1.00  2.59    0.25  0.09   auth
ok       billing    normal 0.98  1.28    0.30  0.17   billing
ok       shipping   normal 0.95  2.32    0.47  0.15   shipping
ok       product    normal 0.99  1.04    0.26  0.21   product
ok       billing    normal 1.00  1.07    0.08  0.07   billing
ok       auth       normal 0.95  2.19    0.25  0.15   auth
ok       product    high   0.91  0.80    0.67  0.43   product
ok       product    normal 0.99  1.21    0.55  0.41   product
ok       billing    normal 0.79  1.04    0.30  0.29   billing
ok       shipping   normal 1.00  1.46    0.05  0.11   shipping
ok       auth       normal 1.00  2.92    0.16  0.13   auth
ok       shipping   high   0.78  0.96    0.67  0.52   shipping

threshold 0.7:
  auto 12/12 (100%), accuracy on auto 100%

threshold sweep (accuracy is only over tickets the model routes itself):
  0.00  coverage 100%  accuracy 100%
  0.50  coverage 100%  accuracy 1